In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Synthetic data mirroring Ethan's scale: ~100 samples, 9 features, binary target.
# We use synthetic data (not real leakage-prone features) specifically so we're
# isolating ONLY the training-methodology difference, not the leakage issue.
X, y = make_classification(
    n_samples=100, n_features=9, n_informative=6, n_redundant=2,
    n_clusters_per_class=2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [2]:
def build_model():
    return MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8),
        activation='relu', solver='adam', alpha=0.001,
        max_iter=1, random_state=42, learning_rate_init=0.001,
        batch_size=16, warm_start=True
    )

In [3]:
model = build_model()
model.fit(X_train, y_train)  # initial fit — only 1 iteration, since max_iter=1

n_val = max(1, int(len(X_train) * 0.2))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

best_val_acc, patience, patience_counter = 0, 20, 0
for epoch in range(100):
    perm = rng.permutation(len(X_tr))
    X_tr, y_tr = X_tr[perm], y_tr[perm]
    for i in range(0, len(X_tr), 16):
        model.partial_fit(X_tr[i:i+16], y_tr[i:i+16])  # partial_fit ignores max_iter
    val_acc = accuracy_score(y_val, model.predict(X_val))
    if val_acc > best_val_acc:
        best_val_acc, patience_counter = val_acc, 0
    else:
        patience_counter += 1
    if patience_counter >= patience:
        break

test_acc_partial_fit_loop = accuracy_score(y_test, model.predict(X_test))
print(f"Path A - partial_fit training loop, held-out test accuracy: {test_acc_partial_fit_loop:.4f}")

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Path A - partial_fit training loop, held-out test accuracy: 0.8000


In [4]:
fresh_model = build_model()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(fresh_model, X, y, cv=skf, scoring='accuracy')

print(f"Path B - cross_val_score fold scores: {cv_scores}")
print(f"Path B - mean CV accuracy: {cv_scores.mean():.4f}")

Path B - cross_val_score fold scores: [0.5  0.6  0.65 0.7  0.5 ]
Path B - mean CV accuracy: 0.5900


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

In [5]:
print(f"\nGap: Path A ({test_acc_partial_fit_loop:.4f}) vs Path B ({cv_scores.mean():.4f})")
print("If Path B is near chance-level (~0.5) while Path A is high,")
print("the training-methodology bug is confirmed as the mechanism.")


Gap: Path A (0.8000) vs Path B (0.5900)
If Path B is near chance-level (~0.5) while Path A is high,
the training-methodology bug is confirmed as the mechanism.
